# 06: Per Cell Line Training

**Purpose:** Train individual RF + XGB models for each of 60 cell lines.

## Cell 2 — Imports

In [2]:
import numpy as np
import pandas as pd
import pickle
import json
import os
import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb

os.makedirs("models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

print("✅ Imports done")
print(f"XGBoost version : {xgb.__version__}")

✅ Imports done
XGBoost version : 3.2.0


## Cell 3 — Load Augmented Data

In [3]:
X_aug     = np.load("data/X_aug.npy")
y_aug     = np.load("data/y_aug.npy")
cells_aug = np.load("data/cells_aug.npy")

unique_cells = sorted(np.unique(cells_aug))

print(f"X_aug shape      : {X_aug.shape}")
print(f"y_aug shape      : {y_aug.shape}")
print(f"Total cell lines : {len(unique_cells)}")

X_aug shape      : (597366, 526)
y_aug shape      : (597366,)
Total cell lines : 60


## Cell 4 — Define Metrics Function

In [4]:
def compute_metrics(y_true, y_pred):
    rp, _  = pearsonr(y_true, y_pred)
    rs, _  = spearmanr(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"Rp": round(float(rp),4), "Rs": round(float(rs),4),
            "R2": round(float(r2),4), "RMSE": round(float(rmse),4)}

print("✅ Metrics function ready")

✅ Metrics function ready


## Cell 5 — Define Reliability Function

In [5]:
def compute_reliability(rf_model, X_test, y_test):
    tree_preds = np.array([tree.predict(X_test)
                           for tree in rf_model.estimators_]).T
    tree_sd    = tree_preds.std(axis=1)
    sorted_idx = np.argsort(tree_sd)
    n          = len(y_test)
    top25      = sorted_idx[:n//4]
    bot25      = sorted_idx[-(n//4):]

    rmse_all   = np.sqrt(mean_squared_error(y_test,
                         rf_model.predict(X_test)))
    rmse_top25 = np.sqrt(mean_squared_error(y_test[top25],
                         rf_model.predict(X_test[top25])))
    rmse_bot25 = np.sqrt(mean_squared_error(y_test[bot25],
                         rf_model.predict(X_test[bot25])))

    return {
        "tree_sd_mean"  : round(float(tree_sd.mean()), 4),
        "rmse_all"      : round(rmse_all, 4),
        "rmse_top25_reliable"   : round(rmse_top25, 4),
        "rmse_bot25_unreliable" : round(rmse_bot25, 4)
    }

print("✅ Reliability function ready")

✅ Reliability function ready


## Cell 6 — RF Training Loop

In [6]:
rf_results = {}

print("=" * 60)
print("PHASE 1 — Random Forest (250 trees) per cell line")
print("=" * 60)

for cellname in tqdm(unique_cells, desc="RF Training"):
    mask    = cells_aug == cellname
    X_cell  = X_aug[mask]
    y_cell  = y_aug[mask]

    X_train, X_test, y_train, y_test = train_test_split(
        X_cell, y_cell, test_size=0.1, random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=250,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)

    y_pred       = rf.predict(X_test)
    metrics      = compute_metrics(y_test, y_pred)
    reliability  = compute_reliability(rf, X_test, y_test)

    safe_name = cellname.replace("/", "_").replace(" ", "_")
    with open(f"models/rf_{safe_name}.pkl", "wb") as f:
        pickle.dump(rf, f)

    rf_results[cellname] = {
        "n_train"     : len(X_train),
        "n_test"      : len(X_test),
        "metrics"     : metrics,
        "reliability" : reliability
    }

print("\n✅ RF Training complete for all 60 cell lines")

PHASE 1 — Random Forest (250 trees) per cell line


RF Training: 100%|██████████| 60/60 [09:32<00:00,  9.55s/it]


✅ RF Training complete for all 60 cell lines


## Cell 7 — RF Results Summary

In [7]:
print("\n=== RF Results Per Cell Line ===\n")
print(f"{'Cell Line':<25} {'Rp':>6} {'Rs':>6} {'R2':>6} {'RMSE':>8}")
print("-" * 55)

rp_values = []
for cell, res in sorted(rf_results.items()):
    m  = res["metrics"]
    rp = m["Rp"]
    rp_values.append(rp)
    print(f"{cell:<25} {m['Rp']:>6.3f} {m['Rs']:>6.3f} "
          f"{m['R2']:>6.3f} {m['RMSE']:>8.3f}")

print("-" * 55)
print(f"\nRF Summary:")
print(f"  Median Rp : {np.median(rp_values):.4f}")
print(f"  Mean Rp   : {np.mean(rp_values):.4f}")


=== RF Results Per Cell Line ===

Cell Line                     Rp     Rs     R2     RMSE
-------------------------------------------------------
786-0                      0.821  0.825  0.668  108.154
A498                       0.731  0.814  0.533  145.192
A549/ATCC                  0.854  0.854  0.723  104.338
ACHN                       0.854  0.862  0.723  115.484
BT-549                     0.820  0.825  0.672  130.847
CAKI-1                     0.800  0.789  0.633  129.347
CCRF-CEM                   0.870  0.869  0.749  181.413
COLO 205                   0.843  0.847  0.703  125.176
DU-145                     0.816  0.830  0.664  128.776
EKVX                       0.833  0.784  0.684   88.421
HCC-2998                   0.819  0.824  0.670  124.157
HCT-116                    0.858  0.873  0.726  112.098
HCT-15                     0.799  0.793  0.634  105.239
HL-60(TB)                  0.835  0.833  0.692  201.044
HOP-62                     0.819  0.822  0.665  131.206
HOP-92       

## Cell 8 — XGB Training Loop

In [8]:
xgb_results = {}

print("=" * 60)
print("PHASE 2 — XGBoost (GPU) per cell line")
print("=" * 60)

XGB_PARAMS = {
    "device"          : "cuda",
    "tree_method"     : "hist",
    "n_estimators"    : 500,
    "learning_rate"   : 0.05,
    "max_depth"       : 6,
    "subsample"       : 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "random_state"    : 42
}

for cellname in tqdm(unique_cells, desc="XGB Training"):
    mask    = cells_aug == cellname
    X_cell  = X_aug[mask]
    y_cell  = y_aug[mask]

    X_train, X_test, y_train, y_test = train_test_split(
        X_cell, y_cell, test_size=0.1, random_state=42
    )

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

    y_pred  = model.predict(X_test)
    metrics = compute_metrics(y_test, y_pred)

    safe_name = cellname.replace("/", "_").replace(" ", "_")
    model.save_model(f"models/xgb_{safe_name}.json")

    xgb_results[cellname] = {
        "n_train" : len(X_train),
        "n_test"  : len(X_test),
        "metrics" : metrics
    }

print("\n✅ XGB Training complete for all 60 cell lines")

PHASE 2 — XGBoost (GPU) per cell line


XGB Training: 100%|██████████| 60/60 [03:13<00:00,  3.22s/it]


✅ XGB Training complete for all 60 cell lines


## Cell 9 — XGB Results Summary

In [9]:
print("\n=== XGB Results Per Cell Line ===\n")
print(f"{'Cell Line':<25} {'Rp':>6} {'Rs':>6} {'R2':>6} {'RMSE':>8}")
print("-" * 55)

xgb_rp_values = []
for cell, res in sorted(xgb_results.items()):
    m  = res["metrics"]
    rp = m["Rp"]
    xgb_rp_values.append(rp)
    print(f"{cell:<25} {m['Rp']:>6.3f} {m['Rs']:>6.3f} "
          f"{m['R2']:>6.3f} {m['RMSE']:>8.3f}")

print("-" * 55)
print(f"\nXGB Summary:")
print(f"  Median Rp : {np.median(xgb_rp_values):.4f}")
print(f"  Mean Rp   : {np.mean(xgb_rp_values):.4f}")


=== XGB Results Per Cell Line ===

Cell Line                     Rp     Rs     R2     RMSE
-------------------------------------------------------
786-0                      0.826  0.844  0.681  106.025
A498                       0.738  0.817  0.541  143.837
A549/ATCC                  0.878  0.874  0.771   94.841
ACHN                       0.878  0.882  0.770  105.187
BT-549                     0.836  0.844  0.698  125.566
CAKI-1                     0.824  0.814  0.679  120.930
CCRF-CEM                   0.879  0.879  0.772  172.849
COLO 205                   0.863  0.868  0.744  116.216
DU-145                     0.836  0.849  0.699  121.880
EKVX                       0.859  0.798  0.736   80.864
HCC-2998                   0.841  0.848  0.705  117.406
HCT-116                    0.877  0.896  0.767  103.222
HCT-15                     0.824  0.811  0.679   98.595
HL-60(TB)                  0.858  0.861  0.736  186.012
HOP-62                     0.839  0.843  0.703  123.498
HOP-92      

## Cell 10 — RF vs XGB Comparison

In [10]:
print("\n=== RF vs XGB — Head to Head ===\n")
print(f"{'Cell Line':<25} {'RF Rp':>7} {'XGB Rp':>7} {'Winner':>8}")
print("-" * 52)

rf_wins  = 0
xgb_wins = 0

for cell in sorted(rf_results.keys()):
    rf_rp  = rf_results[cell]["metrics"]["Rp"]
    xgb_rp = xgb_results[cell]["metrics"]["Rp"]
    winner = "XGB ✅" if xgb_rp > rf_rp else "RF  ✅"
    if xgb_rp > rf_rp:
        xgb_wins += 1
    else:
        rf_wins += 1
    print(f"{cell:<25} {rf_rp:>7.3f} {xgb_rp:>7.3f} {winner:>8}")

print("-" * 52)
print(f"\nRF  wins : {rf_wins}")
print(f"XGB wins : {xgb_wins}")


=== RF vs XGB — Head to Head ===

Cell Line                   RF Rp  XGB Rp   Winner
----------------------------------------------------
786-0                       0.821   0.826    XGB ✅
A498                        0.731   0.738    XGB ✅
A549/ATCC                   0.854   0.878    XGB ✅
ACHN                        0.854   0.878    XGB ✅
BT-549                      0.820   0.836    XGB ✅
CAKI-1                      0.800   0.824    XGB ✅
CCRF-CEM                    0.870   0.879    XGB ✅
COLO 205                    0.843   0.863    XGB ✅
DU-145                      0.816   0.836    XGB ✅
EKVX                        0.833   0.859    XGB ✅
HCC-2998                    0.819   0.841    XGB ✅
HCT-116                     0.858   0.877    XGB ✅
HCT-15                      0.799   0.824    XGB ✅
HL-60(TB)                   0.835   0.858    XGB ✅
HOP-62                      0.819   0.839    XGB ✅
HOP-92                      0.762   0.792    XGB ✅
HS 578T                     0.821   0.853    

## Cell 11 — Save All Results

In [ ]:
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    return obj

all_results = {
    "rf"  : rf_results,
    "xgb" : xgb_results
}

all_results = convert_to_native(all_results)

with open("../results/all_metrics.json", "w") as f:
    json.dump(all_results, f, indent=2)

rows = []
for cell in sorted(rf_results.keys()):
    rows.append({
        "cell_line"  : cell,
        "rf_rp"      : rf_results[cell]["metrics"]["Rp"],
        "rf_rs"      : rf_results[cell]["metrics"]["Rs"],
        "rf_r2"      : rf_results[cell]["metrics"]["R2"],
        "rf_rmse"    : rf_results[cell]["metrics"]["RMSE"],
        "xgb_rp"     : xgb_results[cell]["metrics"]["Rp"],
        "xgb_rs"     : xgb_results[cell]["metrics"]["Rs"],
        "xgb_r2"     : xgb_results[cell]["metrics"]["R2"],
        "xgb_rmse"   : xgb_results[cell]["metrics"]["RMSE"],
        "rf_rmse_top25_reliable" :
            rf_results[cell]["reliability"]["rmse_top25_reliable"],
        "rf_rmse_bot25_unreliable" :
            rf_results[cell]["reliability"]["rmse_bot25_unreliable"],
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv("../results/all_metrics_summary.csv", index=False)

print("✅ Saved → results/all_metrics.json")
print("✅ Saved → results/all_metrics_summary.csv")
print(f"\n{'='*40}")
print("✅ STEP 6 COMPLETE — Ready for 07_results_aggregation.ipynb")
print(f"{'='*40}")

✅ Saved → results/all_metrics.json
✅ Saved → results/all_metrics_summary.csv

✅ STEP 6 COMPLETE — Ready for 07_results_aggregation.ipynb


: 